#### Score Lgr5+/- signature genes from different studies to fetal stem cells
- **Developed by:** Anna Maguza
- **Affilation:** Faculty of Medicine, Würzburg University
- **Date of creation:** 15th November 2024
- **Last modified date:** 15th November 2024


+ Import packages

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import json

### Load fetal data

+ Load fetal cells

In [2]:
fetal_adata = sc.read_h5ad("/mnt/LaCIE/annaM/gut_project/Processed_data/Fetal_stem_cells/Fetal_cells_filtered_raw.h5ad")

+ Normalize fetal cells

In [3]:
fetal_adata_log = fetal_adata.copy()
sc.pp.normalize_total(fetal_adata_log, target_sum = 1e6, exclude_highly_expressed = True)
sc.pp.log1p(fetal_adata_log)

In [4]:
fetal_adata_log = fetal_adata_log[fetal_adata_log.obs['Cell States'].isin(['MTRNR2L12+ASS1+_SC', 'TA', 'RPS10+_RPS17+_SC',
                                                                           'FXYD3+_CKB+_SC', 'Enterocyte'])]

In [5]:
fetal_adata_log_copy = fetal_adata_log.copy()

### Load set of genes

+ We will work with many mouse set of genes, therefore we also need human-mouse orthologues

In [6]:
orthologues_ensembl = pd.read_csv("database_data/ensembl_data/mouse_to_human_orthologues_ensembl.txt", sep = "\t")
orthologues_ensembl[:2]

,Gene stable ID,Gene stable ID version,Transcript stable ID,Transcript stable ID version,Human gene stable ID,Human gene name,Human homology type,%id. target Human gene identical to query gene,%id. query gene identical to target Human gene,Human Gene-order conservation score,"Human orthology confidence [0 low, 1 high]",Human protein or transcript stable ID,Query protein or transcript ID,Protein stable ID,Protein stable ID version,Gene name
0,ENSMUSG00000064341,ENSMUSG00000064341.1,ENSMUST00000082392,ENSMUST00000082392.1,ENSG00000198888,MT-ND1,ortholog_one2one,77.0440,77.0440,50.0,1,ENSP00000354687,ENSMUSP00000080991,ENSMUSP00000080991,ENSMUSP00000080991.1,mt-Nd1
1,ENSMUSG00000064345,ENSMUSG00000064345.1,ENSMUST00000082396,ENSMUST00000082396.1,ENSG00000198763,MT-ND2,ortholog_one2one,57.3913,57.0605,75.0,1,ENSP00000355046,ENSMUSP00000080992,ENSMUSP00000080992,ENSMUSP00000080992.1,mt-Nd2


In [7]:
macrogenes = pd.read_csv("data/Haber_2017_Smartseq/macrogenes_derived_human_mouse_orthologues_10K.csv")
macrogenes[:2]

,Macrogene,Top Human Gene,Top Mouse Gene
0,0,PAK6,Pak4
1,1,PPFIA2,Ppfia4


## Score signature genes to fetal cells

#### Muñoz et al, 2012

In [8]:
munoz_signatures= pd.read_excel("data/Munoz_2012/embj2012166-sup-0002.xls", sheet_name = "S12")
munoz_signatures = munoz_signatures.iloc[2:]
munoz_signatures = munoz_signatures.rename(columns = {munoz_signatures.columns[1]: 'Description'})
munoz_signatures[:2]

,The intestinal stem cell signature,Description,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
2,Wee1,"enriched in Mass Spec, enriched in either Agil...",Wee1,Wee1,Wee1,1.661935,0.07435,1.274753,0.219348,not detected in daughter cell,1.468344
3,Tle3,"enriched in Mass Spec, enriched in either Agil...",Tle3,Tle3,Tle3,0.284206,13.789384,0.983366,0.086606,1.5,1.241683


+ Add human orthologues

In [9]:
munoz_signatures = pd.merge(munoz_signatures, orthologues_ensembl, left_on='The intestinal stem cell signature', right_on='Gene name', how='left')
munoz_signatures[:2]

,The intestinal stem cell signature,Description,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Human homology type,%id. target Human gene identical to query gene,%id. query gene identical to target Human gene,Human Gene-order conservation score,"Human orthology confidence [0 low, 1 high]",Human protein or transcript stable ID,Query protein or transcript ID,Protein stable ID,Protein stable ID version,Gene name
0,Wee1,"enriched in Mass Spec, enriched in either Agil...",Wee1,Wee1,Wee1,1.661935,0.07435,1.274753,0.219348,not detected in daughter cell,...,ortholog_one2one,90.2477,90.2477,100.0,1.0,ENSP00000402084,ENSMUSP00000033326,ENSMUSP00000033326,ENSMUSP00000033326.9,Wee1
1,Tle3,"enriched in Mass Spec, enriched in either Agil...",Tle3,Tle3,Tle3,0.284206,13.789384,0.983366,0.086606,1.5,...,ortholog_one2one,97.3146,98.9597,100.0,1.0,ENSP00000394717,ENSMUSP00000124131,ENSMUSP00000125049,ENSMUSP00000125049.2,Tle3


In [10]:
munoz_signatures['Description'].value_counts()

Description
enriched in Mass Spec, enriched in either Agilent, Affymetrix or both    1000
significant in Affy, enriched in Agilent                                  690
overlap Agilent Affymetrix                                                658
significant in Agilent, enriched in Affy                                  244
overlap Agilent Affymetrix MassSpec                                       240
significant in Affy, enriched in Agilent and MassSpec                     224
significant in Agilent, enriched in Affy and MassSpec                      34
significant in Affy, enriched in MassSpec                                  18
significant in Agilent, enriched in MassSpec                                8
Name: count, dtype: int64

In [11]:
munoz_signatures_enrinched_mass_spec_and_transcriptomics = munoz_signatures[munoz_signatures['Description'] == 'enriched in Mass Spec, enriched in either Agilent, Affymetrix or both']

+ Extract list of genes

In [12]:
munoz_signatures_genes = munoz_signatures['Human gene name'].tolist()
munoz_signatures_genes = [x for x in munoz_signatures_genes if str(x) != 'nan']
munoz_signatures_genes = list(set(munoz_signatures_genes))
len(munoz_signatures_genes)

454

In [13]:
munoz_signatures_enrinched_mass_spec_and_transcriptomics = munoz_signatures_enrinched_mass_spec_and_transcriptomics['Human gene name'].tolist()
munoz_signatures_enrinched_mass_spec_and_transcriptomics = [x for x in munoz_signatures_enrinched_mass_spec_and_transcriptomics if str(x) != 'nan']
munoz_signatures_enrinched_mass_spec_and_transcriptomics = list(set(munoz_signatures_enrinched_mass_spec_and_transcriptomics))
len(munoz_signatures_enrinched_mass_spec_and_transcriptomics)

135

+ Score list of genes

In [14]:
sc.tl.score_genes(fetal_adata_log, gene_list=munoz_signatures_genes, score_name='Muñoz_Lgr5+_score')

       'DACH1', 'GTF2I', 'GPRASP3'],
      dtype='object')


/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:169: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(


In [15]:
sc.tl.score_genes(fetal_adata_log, gene_list=munoz_signatures_enrinched_mass_spec_and_transcriptomics, score_name='Muñoz_Lgr5+_score_enrinched_mass_spec_and_transcriptomics')

#### Haber et al, 2017

In [16]:
haber_bayes = pd.read_csv("data/Haber_2017_Smartseq/gut_mm_DEG_Haber2017_LGR5+_vs_Haber2017_LGR5-_bayes_AM_15112024_145431_DEGs.csv", sep = ",")
haber_bayes[:2]

,Unnamed: 0,key_0,proba_de,proba_not_de,bayes_factor,scale1,scale2,pseudocounts,delta,lfc_mean,...,raw_mean2,non_zeros_proportion1,non_zeros_proportion2,raw_normalized_mean1,raw_normalized_mean2,is_de_fdr_0.05,comparison,group1,group2,gene_name
0,ENSMUSG00000031885,ENSMUSG00000031885,1.0,0.0,18.420681,0.000167,0.000020,0.0,0.25,3.369406,...,19.632479,0.036145,0.401709,1.720034,0.217481,True,LGR5+ stem cell_Grun et al. 2016 vs LGR5- stem...,LGR5+ stem cell_Grun et al. 2016,LGR5- stem cell_Haber et al. 2017,Cbfb
1,ENSMUSG00000000303,ENSMUSG00000000303,1.0,0.0,18.420681,0.004833,0.001256,0.0,0.25,2.042784,...,1162.880342,0.638554,1.000000,51.986577,12.012977,True,LGR5+ stem cell_Grun et al. 2016 vs LGR5- stem...,LGR5+ stem cell_Grun et al. 2016,LGR5- stem cell_Haber et al. 2017,Cdh1


+ Add human orthologues

In [17]:
haber_bayes = pd.merge(haber_bayes, orthologues_ensembl, left_on='gene_name', right_on='Gene name', how='left')
haber_bayes[:2]

,Unnamed: 0,key_0,proba_de,proba_not_de,bayes_factor,scale1,scale2,pseudocounts,delta,lfc_mean,...,Human homology type,%id. target Human gene identical to query gene,%id. query gene identical to target Human gene,Human Gene-order conservation score,"Human orthology confidence [0 low, 1 high]",Human protein or transcript stable ID,Query protein or transcript ID,Protein stable ID,Protein stable ID version,Gene name
0,ENSMUSG00000031885,ENSMUSG00000031885,1.0,0.0,18.420681,0.000167,0.00002,0.0,0.25,3.369406,...,ortholog_one2one,98.9305,98.9305,100.0,1.0,ENSP00000415151,ENSMUSP00000059382,ENSMUSP00000059382,ENSMUSP00000059382.3,Cbfb
1,ENSMUSG00000031885,ENSMUSG00000031885,1.0,0.0,18.420681,0.000167,0.00002,0.0,0.25,3.369406,...,ortholog_one2one,98.9305,98.9305,100.0,1.0,ENSP00000415151,ENSMUSP00000059382,ENSMUSP00000105019,ENSMUSP00000105019.2,Cbfb


+ Extract top 100 genes based on the bayes factors

In [18]:
haber_bayes = haber_bayes.drop_duplicates(subset = 'Human gene name')
haber_bayes = haber_bayes.nlargest(100, 'bayes_factor')

+ Extract genes to list

In [19]:
haber_bayes_genes = haber_bayes['Human gene name'].tolist()
haber_bayes_genes = [x for x in haber_bayes_genes if str(x) != 'nan']
haber_bayes_genes = list(set(haber_bayes_genes))
len(haber_bayes_genes)

99

+ Score list of genes

In [20]:
sc.tl.score_genes(fetal_adata_log, gene_list=haber_bayes_genes, score_name='Haber_Lgr5+_bayes_factors')

       'PRAMEF8', 'PATE4', 'PRAMEF6', 'PRAMEF5', 'PRAMEF19', 'PRAMEF17',
       'PRAMEF11', 'PRAMEF18', 'PRAMEF20', 'PRAMEF7', 'PRAMEF26', 'PRAMEF12',
       'PRAMEF27', 'PRAMEF15', 'PRAMEF1', 'PRAMEF33', 'PRAMEF14', 'SOCS7',
       'PRAMEF2'],
      dtype='object')


#### Repeat the same with wilcoxon-derived genes

In [21]:
haber_wilcoxon = pd.read_csv("data/Haber_2017_Smartseq/gut_mm_DEG_Haber2017_LGR5+_vs_Haber2017_LGR5-_wilcoxon_AM_15112024_145431_DEGs.csv", sep = ",")
haber_wilcoxon[:2]

,Unnamed: 0,group,names,scores,logfoldchanges,pvals,pvals_adj
0,0,LGR5+ stem cell_Haber et al. 2017,ENSMUSG00000031320,8.165997,0.459153,3.187921e-16,4.910036e-12
1,1,LGR5+ stem cell_Haber et al. 2017,ENSMUSG00000020140,7.825796,1.449033,5.044543e-15,3.884802e-11


In [22]:
haber_wilcoxon = pd.merge(haber_wilcoxon, orthologues_ensembl, left_on='names', right_on='Gene stable ID', how='left')
haber_wilcoxon[:2]

,Unnamed: 0,group,names,scores,logfoldchanges,pvals,pvals_adj,Gene stable ID,Gene stable ID version,Transcript stable ID,...,Human homology type,%id. target Human gene identical to query gene,%id. query gene identical to target Human gene,Human Gene-order conservation score,"Human orthology confidence [0 low, 1 high]",Human protein or transcript stable ID,Query protein or transcript ID,Protein stable ID,Protein stable ID version,Gene name
0,0,LGR5+ stem cell_Haber et al. 2017,ENSMUSG00000031320,8.165997,0.459153,3.187921e-16,4.910036e-12,ENSMUSG00000031320,ENSMUSG00000031320.10,ENSMUST00000033683,...,ortholog_one2one,100.0,100.0,100.0,1.0,ENSP00000362744,ENSMUSP00000033683,ENSMUSP00000033683,ENSMUSP00000033683.8,Rps4x
1,0,LGR5+ stem cell_Haber et al. 2017,ENSMUSG00000031320,8.165997,0.459153,3.187921e-16,4.910036e-12,ENSMUSG00000031320,ENSMUSG00000031320.10,ENSMUST00000155028,...,ortholog_one2one,100.0,100.0,100.0,1.0,ENSP00000362744,ENSMUSP00000033683,NaN,NaN,Rps4x


In [23]:
haber_wilcoxon_lgr5high = haber_wilcoxon[haber_wilcoxon['group']=='LGR5+ stem cell_Haber et al. 2017']
haber_wilcoxon_lgr5low = haber_wilcoxon[haber_wilcoxon['group']=='LGR5- stem cell_Haber et al. 2017']

In [24]:
haber_wilcoxon_lgr5high_genes = haber_wilcoxon_lgr5high['Human gene name'].tolist()
haber_wilcoxon_lgr5high_genes = [x for x in haber_wilcoxon_lgr5high_genes if str(x) != 'nan']
haber_wilcoxon_lgr5high_genes = list(set(haber_wilcoxon_lgr5high_genes))
len(haber_wilcoxon_lgr5high_genes)

83

In [25]:
haber_wilcoxon_lgr5low_genes = haber_wilcoxon_lgr5low['Human gene name'].tolist()
haber_wilcoxon_lgr5low_genes = [x for x in haber_wilcoxon_lgr5low_genes if str(x) != 'nan']
haber_wilcoxon_lgr5low_genes = list(set(haber_wilcoxon_lgr5low_genes))
len(haber_wilcoxon_lgr5low_genes)

144

In [26]:
sc.tl.score_genes(fetal_adata_log, gene_list=haber_wilcoxon_lgr5high_genes, score_name='Haber_Lgr5+_wilcoxon')
sc.tl.score_genes(fetal_adata_log, gene_list=haber_wilcoxon_lgr5low_genes, score_name='Haber_Lgr5-_wilcoxon')

### Grün et al, 2016

In [27]:
grun_bayes = pd.read_csv("data/Haber_2017_Smartseq/gut_mm_Grün2016_LGR5+_vs_Haber2017_LGR5-_bayes_AM_15112024_145431_DEGs.csv", sep = ",")
grun_bayes[:2]

,Unnamed: 0,key_0,proba_de,proba_not_de,bayes_factor,scale1,scale2,pseudocounts,delta,lfc_mean,...,raw_mean2,non_zeros_proportion1,non_zeros_proportion2,raw_normalized_mean1,raw_normalized_mean2,is_de_fdr_0.05,comparison,group1,group2,gene_name
0,ENSMUSG00000031885,ENSMUSG00000031885,1.0,0.0,18.420681,0.000167,0.000020,0.0,0.25,3.369406,...,19.632479,0.036145,0.401709,1.720034,0.217481,True,LGR5+ stem cell_Grun et al. 2016 vs LGR5- stem...,LGR5+ stem cell_Grun et al. 2016,LGR5- stem cell_Haber et al. 2017,Cbfb
1,ENSMUSG00000000303,ENSMUSG00000000303,1.0,0.0,18.420681,0.004833,0.001256,0.0,0.25,2.042784,...,1162.880342,0.638554,1.000000,51.986577,12.012977,True,LGR5+ stem cell_Grun et al. 2016 vs LGR5- stem...,LGR5+ stem cell_Grun et al. 2016,LGR5- stem cell_Haber et al. 2017,Cdh1


+ Add human orthologues

In [28]:
grun_bayes = pd.merge(grun_bayes, orthologues_ensembl, left_on='gene_name', right_on='Gene name', how='left')
grun_bayes[:2]

,Unnamed: 0,key_0,proba_de,proba_not_de,bayes_factor,scale1,scale2,pseudocounts,delta,lfc_mean,...,Human homology type,%id. target Human gene identical to query gene,%id. query gene identical to target Human gene,Human Gene-order conservation score,"Human orthology confidence [0 low, 1 high]",Human protein or transcript stable ID,Query protein or transcript ID,Protein stable ID,Protein stable ID version,Gene name
0,ENSMUSG00000031885,ENSMUSG00000031885,1.0,0.0,18.420681,0.000167,0.00002,0.0,0.25,3.369406,...,ortholog_one2one,98.9305,98.9305,100.0,1.0,ENSP00000415151,ENSMUSP00000059382,ENSMUSP00000059382,ENSMUSP00000059382.3,Cbfb
1,ENSMUSG00000031885,ENSMUSG00000031885,1.0,0.0,18.420681,0.000167,0.00002,0.0,0.25,3.369406,...,ortholog_one2one,98.9305,98.9305,100.0,1.0,ENSP00000415151,ENSMUSP00000059382,ENSMUSP00000105019,ENSMUSP00000105019.2,Cbfb


+ Extract top 100 genes based on the bayes factors

In [29]:
grun_bayes = grun_bayes.drop_duplicates(subset = 'Human gene name')
grun_bayes = grun_bayes.nlargest(100, 'bayes_factor')

+ Extract genes to list

In [30]:
grun_bayes_genes = grun_bayes['Human gene name'].tolist()
grun_bayes_genes = [x for x in grun_bayes_genes if str(x) != 'nan']
grun_bayes_genes = list(set(grun_bayes_genes))
len(grun_bayes_genes)

99

+ Score list of genes

In [31]:
sc.tl.score_genes(fetal_adata_log, gene_list=grun_bayes_genes, score_name='Grün_Lgr5+_bayes_factors')

       'PRAMEF8', 'PATE4', 'PRAMEF6', 'PRAMEF5', 'PRAMEF19', 'PRAMEF17',
       'PRAMEF11', 'PRAMEF18', 'PRAMEF20', 'PRAMEF7', 'PRAMEF26', 'PRAMEF12',
       'PRAMEF27', 'PRAMEF15', 'PRAMEF1', 'PRAMEF33', 'PRAMEF14', 'SOCS7',
       'PRAMEF2'],
      dtype='object')


#### Try the same with wilcoxon-derived genes

In [32]:
grun_wilcoxon = pd.read_csv("data/Haber_2017_Smartseq/gut_mm_DEG_Grün2016_LGR5+_vs_Haber2017_LGR5-_wilcoxon_AM_15112024_145431_DEGs.csv", sep = ",")
grun_bayes[:2]

,Unnamed: 0,key_0,proba_de,proba_not_de,bayes_factor,scale1,scale2,pseudocounts,delta,lfc_mean,...,Human homology type,%id. target Human gene identical to query gene,%id. query gene identical to target Human gene,Human Gene-order conservation score,"Human orthology confidence [0 low, 1 high]",Human protein or transcript stable ID,Query protein or transcript ID,Protein stable ID,Protein stable ID version,Gene name
0,ENSMUSG00000031885,ENSMUSG00000031885,1.0,0.0,18.420681,0.000167,0.000020,0.0,0.25,3.369406,...,ortholog_one2one,98.9305,98.9305,100.0,1.0,ENSP00000415151,ENSMUSP00000059382,ENSMUSP00000059382,ENSMUSP00000059382.3,Cbfb
7,ENSMUSG00000000303,ENSMUSG00000000303,1.0,0.0,18.420681,0.004833,0.001256,0.0,0.25,2.042784,...,ortholog_one2one,80.8824,81.0658,100.0,1.0,ENSP00000261769,ENSMUSP00000000312,ENSMUSP00000000312,ENSMUSP00000000312.6,Cdh1


+ Add human orthologues

In [33]:
grun_wilcoxon = pd.merge(grun_wilcoxon, orthologues_ensembl, left_on='names', right_on='Gene stable ID', how='left')
grun_wilcoxon[:2]

,Unnamed: 0,group,names,scores,logfoldchanges,pvals,pvals_adj,Gene stable ID,Gene stable ID version,Transcript stable ID,...,Human homology type,%id. target Human gene identical to query gene,%id. query gene identical to target Human gene,Human Gene-order conservation score,"Human orthology confidence [0 low, 1 high]",Human protein or transcript stable ID,Query protein or transcript ID,Protein stable ID,Protein stable ID version,Gene name
0,0,LGR5+ stem cell_Grun et al. 2016,ENSMUSG00000026238,14.645286,2.711531,1.443934e-48,7.488037e-47,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,LGR5+ stem cell_Grun et al. 2016,ENSMUSG00000064345,9.684206,0.964496,3.519322e-22,1.118082e-21,ENSMUSG00000064345,ENSMUSG00000064345.1,ENSMUST00000082396,...,ortholog_one2one,57.3913,57.0605,75.0,1.0,ENSP00000355046,ENSMUSP00000080992,ENSMUSP00000080992,ENSMUSP00000080992.1,mt-Nd2


In [34]:
grun_wilcoxon_lgr5high = grun_wilcoxon[grun_wilcoxon['group']=='LGR5+ stem cell_Grun et al. 2016']

In [35]:
grun_wilcoxon_lgr5high_genes = grun_wilcoxon_lgr5high['Human gene name'].tolist()
grun_wilcoxon_lgr5high_genes = [x for x in grun_wilcoxon_lgr5high_genes if str(x) != 'nan']
grun_wilcoxon_lgr5high_genes = list(set(grun_wilcoxon_lgr5high_genes))
len(grun_wilcoxon_lgr5high_genes)

119

In [36]:
sc.tl.score_genes(fetal_adata_log, gene_list=grun_wilcoxon_lgr5high_genes, score_name='Grün_Lgr5+_wilcoxon')

       'RN7SL116P', 'LY6G6F', 'CCL23'],
      dtype='object')


### Ishikawa et al, 2022

In [37]:
ishikawa_wilcoxon = pd.read_csv("data/Ishikawa_2022/gut_hs_Ishikawa2022_LGR5+vsLGR5-_DEGs_wilcoxon_AM_15112024_142050.csv", sep = ",")
ishikawa_wilcoxon[:2]

,Unnamed: 0,group,names,scores,logfoldchanges,pvals,pvals_adj
0,0,LGR5_high_stem_cell,LGR5,8.559304,36.761272,1.135511e-17,3.808277e-13
1,1,LGR5_high_stem_cell,RPL13A,3.069295,0.122809,2.145646e-03,1.000000e+00


In [38]:
ishikawa_wilcoxon_lgr5high = ishikawa_wilcoxon[ishikawa_wilcoxon['group']=='LGR5_high_stem_cell']
ishikawa_wilcoxon_lgr5low = ishikawa_wilcoxon[ishikawa_wilcoxon['group']=='LGR5_low_stem_cell']

In [39]:
ishikawa_wilcoxon_lgr5high_genes = ishikawa_wilcoxon_lgr5high['names'].tolist()
ishikawa_wilcoxon_lgr5high_genes = [x for x in ishikawa_wilcoxon_lgr5high_genes if str(x) != 'nan']
ishikawa_wilcoxon_lgr5high_genes = list(set(ishikawa_wilcoxon_lgr5high_genes))
len(ishikawa_wilcoxon_lgr5high_genes)

100

In [40]:
ishikawa_wilcoxon_lgr5low_genes = ishikawa_wilcoxon_lgr5low['names'].tolist()
ishikawa_wilcoxon_lgr5low_genes = [x for x in ishikawa_wilcoxon_lgr5low_genes if str(x) != 'nan']
ishikawa_wilcoxon_lgr5low_genes = list(set(ishikawa_wilcoxon_lgr5low_genes))
len(ishikawa_wilcoxon_lgr5low_genes)

100

In [41]:
sc.tl.score_genes(fetal_adata_log, gene_list=ishikawa_wilcoxon_lgr5high_genes, score_name='Ishikawa_Lgr5+_wilcoxon')
sc.tl.score_genes(fetal_adata_log, gene_list=ishikawa_wilcoxon_lgr5low_genes, score_name='Ishikawa_Lgr5-_wilcoxon')

In [43]:
fetal_adata_log.obs_keys

<bound method AnnData.obs_keys of AnnData object with n_obs × n_vars = 28981 × 25445
    obs: 'Sample_ID', 'Cell Type', 'Study_name', 'Donor_ID', 'Diagnosis', 'Age', 'Region code', 'Fraction', 'Sex', 'Library_Preparation_Protocol', 'batch', 'Age_group', 'Location', 'Cell States', 'Cell States GCA', 'Chem', 'Layer', 'Cell States Kong', 'dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'pct_counts_ribo', 'Cell_ID', '_scvi_batch', '_scvi_labels', 'n_genes', 'n_counts', 'Muñoz_Lgr5+_score', 'Muñoz_Lgr5+_score_enrinched_mass_spec_and_transcriptomics', 'Haber_Lgr5+_bayes_factors', 'Haber_Lgr5+_wilcoxon', 'Haber_Lgr5-_wilcoxon', 'Grün_Lgr5+_bayes_factors', 'Grün_Lgr5+_wilcoxon', 'Ishikawa_Lgr5+_wilcoxon', 'Ishikawa_Lgr5-_wilcoxon'
    var: 'feature_types-0-0-0', 'gene_name-1-0-0', 'gene_id-0-0', 'GENE-1-0', 'n_counts', 'n_cells'
    uns: 'log1p'>

In [48]:
sc.set_figure_params(dpi=300, figsize=(5, 5))
sc.pl.violin(fetal_adata_log, keys=['Muñoz_Lgr5+_score', 'Muñoz_Lgr5+_score_enrinched_mass_spec_and_transcriptomics', 'Haber_Lgr5+_bayes_factors', 'Haber_Lgr5+_wilcoxon', 'Haber_Lgr5-_wilcoxon', 'Grün_Lgr5+_bayes_factors', 'Grün_Lgr5+_wilcoxon', 'Ishikawa_Lgr5+_wilcoxon', 'Ishikawa_Lgr5-_wilcoxon'], groupby='Cell States', stripplot=True, inner="quartile", rotation=90)